# Clasificación de 3 escenas con Swin Transformer

Implementación de **Swin Transformer** (*Shifted Window Transformer*, Liu et
al., 2021) en Keras, entrenada **desde cero** sobre el dataset local de 3
escenas (`coast`, `forest`, `highway`; 961 imágenes). Swin introduce dos ideas
clave frente a un ViT clásico:

1. **Atención por ventanas locales.** Un ViT calcula atención global: cada token
   atiende a todos los demás y el coste crece con el cuadrado del número de
   tokens, `O(n²)`. Swin trocea el mapa de tokens en ventanas de 4×4 y calcula
   la atención solo dentro de cada ventana; con un número fijo de tokens por
   ventana, el coste total pasa a ser lineal en el tamaño de la imagen, `O(n)`.
2. **Ventanas desplazadas.** Con ventanas fijas, cada región de la imagen
   quedaría aislada y el modelo no podría combinar información de ventanas
   vecinas. En capas alternas Swin desplaza la retícula media ventana, de modo
   que las nuevas ventanas mezclan tokens que en la capa anterior pertenecían a
   ventanas distintas: la información cruza las fronteras entre ventanas (con
   una máscara de atención que impide atender donde no toca).

La arquitectura es jerárquica: `PatchMerging` reduce la resolución espacial a la
mitad en cada etapa y duplica la dimensión, igual que el *pooling* de una CNN.
Esto importa en este dataset porque **las tres escenas se distinguen a varias
escalas**: la costa por la línea de agua y el horizonte, el bosque por la
textura fina de la vegetación y la autopista por las líneas y bordillos
alargados de la calzada.

Se entrena **desde cero** (sin pesos de ImageNet) para que la comparación separe
*arquitectura* (atención vs convolución) de *preentrenamiento* (ImageNet vs
desde cero): la CNN y este Swin se entrenan los dos desde cero sobre las mismas
961 imágenes, y la DenseNet121 de `ScenesDenseNet.ipynb` parte de ImageNet.

Adaptación al problema: entrada 128×128, parches de 8×8, cuatro etapas con
dimensiones (48, 96, 192, 384), profundidades (2, 2, 2, 2), cabezas (2, 4, 8,
16) y ventanas de 4×4, con las tres clases de `cs.CLASSES` y el split común del
reto (≈ 612 train / 108 validación / 241 test, semilla 20, en
`resultados/split.json`).

El modelo se carga desde `resultados/swin/model.weights.h5` por defecto (las
capas personalizadas no se deserializan con fiabilidad, así que se reconstruye
la arquitectura y se cargan los pesos). Cambia `FORCE_RETRAIN = True` para
entrenarlo de nuevo.


## Configuración


In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from keras import layers, ops
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import AdamW

import common_3scenes as cs

FORCE_RETRAIN = False
sns.set_theme(style="whitegrid", context="notebook")
keras.utils.set_random_seed(cs.SEED)
np.random.seed(cs.SEED)

## Arquitectura Swin

* `window_partition` / `window_reverse` trocean el mapa de tokens en ventanas
  de `window_size`×`window_size` y lo reconstruyen.
* `WindowAttention` es la atención multi-cabeza dentro de una ventana: capas
  `qkv` y `proj`, más una **tabla de sesgo de posición relativa** que aprende
  cuánto atender según la distancia entre tokens.
* `SwinBlock` alterna bloques con ventanas normales (`shift_size = 0`) y
  desplazadas (`shift_size = window_size // 2`); en los desplazados, una
  máscara de atención (`-100` donde dos tokens no compartían ventana original)
  evita que la atención cruce donde no debe.
* `BasicLayer` apila los `depth` bloques de una etapa y `PatchMerging` junta
  parches 2×2 (la resolución baja a la mitad y la dimensión se duplica): cuatro
  etapas, como los bloques de una CNN.
* `build_swin` une todo: `PatchEmbed` (un `Conv2D` con stride igual al tamaño
  del parche) → 4 etapas → `LayerNormalization` + `GlobalAveragePooling1D` +
  `Dropout(0.2)` + `Dense(3, softmax)`.


In [ ]:
def window_partition(x, window_size):
    _, height, width, channels = x.shape
    x = ops.reshape(
        x,
        (-1, height // window_size, window_size, width // window_size, window_size, channels),
    )
    x = ops.transpose(x, (0, 1, 3, 2, 4, 5))
    return ops.reshape(x, (-1, window_size, window_size, channels))


def window_reverse(windows, window_size, height, width):
    channels = windows.shape[-1]
    x = ops.reshape(
        windows,
        (-1, height // window_size, width // window_size, window_size, window_size, channels),
    )
    x = ops.transpose(x, (0, 1, 3, 2, 4, 5))
    return ops.reshape(x, (-1, height, width, channels))


class WindowAttention(layers.Layer):
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.qkv = layers.Dense(dim * 3, use_bias=qkv_bias)
        self.proj = layers.Dense(dim)
        self.dropout = layers.Dropout(dropout)
        self.relative_position_bias_table = self.add_weight(
            shape=((2 * window_size - 1) ** 2, num_heads),
            initializer=keras.initializers.TruncatedNormal(stddev=0.02),
            trainable=True,
            name="relative_position_bias_table",
        )

        coordinates_h = np.arange(window_size)
        coordinates_w = np.arange(window_size)
        coordinates = np.stack(
            np.meshgrid(coordinates_h, coordinates_w, indexing="ij")
        )
        coordinates_flat = coordinates.reshape(2, -1)
        relative = coordinates_flat[:, :, None] - coordinates_flat[:, None, :]
        relative = relative.transpose([1, 2, 0])
        relative[:, :, 0] += window_size - 1
        relative[:, :, 1] += window_size - 1
        relative[:, :, 0] *= 2 * window_size - 1
        self.relative_position_index = ops.convert_to_tensor(
            relative.sum(-1), dtype="int32"
        )

    def call(self, x, attn_mask=None, training=False):
        tokens = x.shape[1]
        head_dim = self.dim // self.num_heads
        qkv = self.qkv(x)
        qkv = ops.reshape(qkv, (-1, tokens, 3, self.num_heads, head_dim))
        qkv = ops.transpose(qkv, (2, 0, 3, 1, 4))
        query, key, value = qkv[0], qkv[1], qkv[2]

        attention = (query * self.scale) @ ops.transpose(key, (0, 1, 3, 2))
        indices = ops.reshape(self.relative_position_index, (-1,))
        bias = ops.take(self.relative_position_bias_table, indices, axis=0)
        bias = ops.reshape(bias, (tokens, tokens, -1))
        bias = ops.transpose(bias, (2, 0, 1))
        attention = attention + bias[None]

        if attn_mask is not None:
            windows = attn_mask.shape[0]
            attention = ops.reshape(
                attention, (-1, windows, self.num_heads, tokens, tokens)
            )
            attention = attention + attn_mask[None, :, None, :, :]
            attention = ops.reshape(attention, (-1, self.num_heads, tokens, tokens))

        attention = ops.softmax(attention, axis=-1)
        attention = self.dropout(attention, training=training)
        output = attention @ value
        output = ops.transpose(output, (0, 2, 1, 3))
        output = ops.reshape(output, (-1, tokens, self.dim))
        return self.dropout(self.proj(output), training=training)


class SwinBlock(layers.Layer):
    def __init__(
        self,
        dim,
        num_heads,
        input_resolution,
        window_size=4,
        shift_size=0,
        mlp_ratio=4.0,
        dropout=0.0,
        drop_path=0.0,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.dim = dim
        self.num_heads = num_heads
        self.height, self.width = input_resolution
        self.window_size = min(window_size, self.height, self.width)
        self.shift_size = (
            0
            if self.window_size <= 1
            else min(shift_size, self.window_size // 2)
        )
        self.norm1 = layers.LayerNormalization(epsilon=1e-5)
        self.attn = WindowAttention(
            dim, self.window_size, num_heads, dropout=dropout
        )
        self.norm2 = layers.LayerNormalization(epsilon=1e-5)
        self.mlp = keras.Sequential(
            [
                layers.Dense(int(dim * mlp_ratio)),
                layers.Activation("gelu"),
                layers.Dropout(dropout),
                layers.Dense(dim),
                layers.Dropout(dropout),
            ]
        )
        self.drop_path = layers.Dropout(drop_path)
        self.attn_mask = None
        if self.shift_size > 0:
            height, width = self.height, self.width
            height_slices = (
                slice(0, -self.window_size),
                slice(-self.window_size, -self.shift_size),
                slice(-self.shift_size, None),
            )
            width_slices = (
                slice(0, -self.window_size),
                slice(-self.window_size, -self.shift_size),
                slice(-self.shift_size, None),
            )
            mask = np.zeros((1, height, width, 1), dtype=np.float32)
            count = 0
            for height_slice in height_slices:
                for width_slice in width_slices:
                    mask[:, height_slice, width_slice, :] = count
                    count += 1
            mask = ops.convert_to_tensor(mask)
            mask_windows = window_partition(mask, self.window_size)
            mask_windows = ops.reshape(
                mask_windows, (-1, self.window_size * self.window_size)
            )
            attention_mask = mask_windows[:, None, :] - mask_windows[:, :, None]
            self.attn_mask = ops.where(attention_mask != 0, -100.0, 0.0)

    def call(self, x, training=False):
        shortcut = x
        x = self.norm1(x)
        x = ops.reshape(x, (-1, self.height, self.width, self.dim))
        if self.shift_size > 0:
            x = ops.roll(
                x, shift=(-self.shift_size, -self.shift_size), axis=(1, 2)
            )
        windows = window_partition(x, self.window_size)
        windows = ops.reshape(
            windows, (-1, self.window_size * self.window_size, self.dim)
        )
        windows = self.attn(windows, attn_mask=self.attn_mask, training=training)
        windows = ops.reshape(
            windows, (-1, self.window_size, self.window_size, self.dim)
        )
        x = window_reverse(windows, self.window_size, self.height, self.width)
        if self.shift_size > 0:
            x = ops.roll(x, shift=(self.shift_size, self.shift_size), axis=(1, 2))
        x = ops.reshape(x, (-1, self.height * self.width, self.dim))
        x = shortcut + self.drop_path(x, training=training)
        x = x + self.drop_path(self.mlp(self.norm2(x)), training=training)
        return x


class BasicLayer(layers.Layer):
    def __init__(
        self,
        dim,
        depth,
        num_heads,
        input_resolution,
        window_size=4,
        drop_path_rate=0.0,
        dropout=0.0,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.blocks = [
            SwinBlock(
                dim,
                num_heads,
                input_resolution,
                window_size,
                shift_size=0 if index % 2 == 0 else window_size // 2,
                dropout=dropout,
                drop_path=drop_path_rate * index / max(depth - 1, 1),
            )
            for index in range(depth)
        ]

    def call(self, x, training=False):
        for block in self.blocks:
            x = block(x, training=training)
        return x


class PatchEmbed(layers.Layer):
    def __init__(self, dim, patch_size, input_size, **kwargs):
        super().__init__(**kwargs)
        self.proj = layers.Conv2D(dim, kernel_size=patch_size, strides=patch_size)
        self.norm = layers.LayerNormalization(epsilon=1e-5)
        self.num_patches = (input_size // patch_size) ** 2
        self.pos_embed = self.add_weight(
            shape=(1, self.num_patches, dim),
            initializer=keras.initializers.TruncatedNormal(stddev=0.02),
            trainable=True,
            name="pos_embed",
        )

    def call(self, x):
        x = self.proj(x)
        _, height, width, channels = x.shape
        x = ops.reshape(x, (-1, height * width, channels))
        return self.norm(x + self.pos_embed)


class PatchMerging(layers.Layer):
    def __init__(self, dim, input_resolution, **kwargs):
        super().__init__(**kwargs)
        self.height, self.width = input_resolution
        self.norm = layers.LayerNormalization(epsilon=1e-5)
        self.reduction = layers.Dense(2 * dim, use_bias=False)

    def call(self, x):
        channels = x.shape[-1]
        x = ops.reshape(x, (-1, self.height, self.width, channels))
        top_left = x[:, 0::2, 0::2, :]
        top_right = x[:, 1::2, 0::2, :]
        bottom_left = x[:, 0::2, 1::2, :]
        bottom_right = x[:, 1::2, 1::2, :]
        x = ops.concatenate(
            [top_left, top_right, bottom_left, bottom_right], axis=-1
        )
        x = ops.reshape(
            x, (-1, (self.height // 2) * (self.width // 2), 4 * channels)
        )
        return self.reduction(self.norm(x))


def build_swin(
    input_size=128,
    num_classes=3,
    dims=(48, 96, 192, 384),
    depths=(2, 2, 2, 2),
    heads=(2, 4, 8, 16),
    patch_size=8,
    window_size=4,
    dropout=0.05,
    drop_path_rate=0.0,
):
    inputs = layers.Input(shape=(input_size, input_size, 3))
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.05)(x)
    x = layers.RandomZoom(0.10)(x)
    x = layers.Rescaling(1.0 / 255.0)(x)
    x = PatchEmbed(dims[0], patch_size, input_size)(x)

    height = width = input_size // patch_size
    for index, (dim, depth, num_heads) in enumerate(zip(dims, depths, heads)):
        x = BasicLayer(
            dim,
            depth,
            num_heads,
            (height, width),
            window_size,
            drop_path_rate=drop_path_rate,
            dropout=dropout,
        )(x)
        if index < len(dims) - 1:
            x = PatchMerging(dim, (height, width))(x)
            height, width = height // 2, width // 2

    x = layers.LayerNormalization(epsilon=1e-5)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="swin_3scenes")

## Datos, entrenamiento y carga desde caché

El split es el común del reto (`cs.load_split_data()`: 961 imágenes → ≈ 612
train / 108 validación / 241 test, `resultados/split.json`); este cuaderno no
hace `train_test_split`.

Entrenar un Transformer **desde cero** con ~612 imágenes de train es el ajuste
más difícil de los cuatro modelos del reto: sin los sesgos inductivos de las
convoluciones (localidad, compartición de pesos), el modelo memoriza el train
en cuanto tiene capacidad para ello. El protocolo replica el de referencia y
añade lo imprescindible:

* **Augmentación** dentro del modelo: `RandomFlip("horizontal")`,
  `RandomRotation(0.05)` y `RandomZoom(0.10)`, solo activa en entrenamiento.
* **Weight decay**: `AdamW(learning_rate=3e-4, weight_decay=0.01)`.
* **EarlyStopping** sobre `val_accuracy` (paciencia 12, `restore_best_weights`)
  y **ReduceLROnPlateau** sobre `val_loss` (factor 0,5, paciencia 5), con un
  máximo de 40 épocas y batch de 32.

El entrenamiento usa `cs.load_or_train` con `load_model=load_swin`: como las
capas personalizadas no se deserializan con fiabilidad, `load_swin` reconstruye
la arquitectura con `build_swin()` y carga `model.weights.h5`. Los resultados
se guardan en `resultados/swin/`, la carpeta que `reto-1.ipynb` espera para
añadir el Transformer a tabla, figuras, McNemar y conclusiones.


In [ ]:
data = cs.load_split_data()
X_train = data["X_train"].astype("float32")
X_val = data["X_val"].astype("float32")
X_test = data["X_test"].astype("float32")
y_train_labels = data["y_train"]
y_val_labels = data["y_val"]
y_test_labels = data["y_test"]
test_paths = data["test_paths"]

y_train = cs.one_hot(y_train_labels)
y_val = cs.one_hot(y_val_labels)

print("Train:", X_train.shape, "| Validación:", X_val.shape, "| Test:", X_test.shape)
print("Distribución de test:", {name: int(np.sum(y_test_labels == name)) for name in cs.CLASSES})

In [ ]:
def fit_swin(model):
    model.compile(
        loss="categorical_crossentropy",
        optimizer=AdamW(learning_rate=3e-4, weight_decay=0.01),
        metrics=["accuracy"],
    )
    callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=12, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    ]
    history, seconds = cs.train_keras_model(
        model,
        X_train,
        y_train,
        X_val,
        y_val,
        epochs=40,
        batch_size=32,
        callbacks=callbacks,
    )
    return model, history, seconds


def evaluate_swin(model, history, training_seconds):
    predicted_idx, probabilities, inference_ms = cs.predict_keras(model, X_test)
    predicted = np.array(cs.CLASSES)[predicted_idx]
    metrics = cs.compute_metrics(
        name="Swin Transformer",
        family="IA generativa (Transformer)",
        y_true=y_test_labels,
        y_pred=predicted,
        y_prob=probabilities,
        params=model.count_params(),
        trainable_params=sum(np.prod(weight.shape) for weight in model.trainable_weights),
        training_seconds=training_seconds,
        inference_ms=inference_ms,
        epochs=len(history.history["loss"]),
        hyperparameters={
            "input": 128,
            "patch_size": 8,
            "embed_dims": [48, 96, 192, 384],
            "depths": [2, 2, 2, 2],
            "num_heads": [2, 4, 8, 16],
            "window_size": 4,
            "dropout": 0.05,
            "optimizer": "AdamW",
            "lr": 3e-4,
            "weight_decay": 0.01,
            "batch_size": 32,
            "epochs": 40,
            "trained_from_scratch": True,
        },
        notes="Swin Transformer trained from scratch on the 3-scene dataset (no ImageNet weights).",
    )
    cs.save_results(
        "swin",
        metrics,
        y_test_labels,
        predicted,
        probabilities,
        test_paths=test_paths,
        history=history,
        model=model,
    )
    # The .keras file contains custom layers, so the weights are also saved
    # separately for a reliable reload.
    model.save_weights(cs.result_dir("swin") / cs.ARTIFACTS["weights"])
    return metrics


def load_swin(folder):
    weights_path = cs.result_dir(folder) / cs.ARTIFACTS["weights"]
    if not weights_path.exists():
        return None
    model = build_swin()
    model.load_weights(weights_path)
    return model

In [ ]:
model, history, metrics, retrained = cs.load_or_train(
    "swin",
    build_model=build_swin,
    fit_model=fit_swin,
    evaluate_model=evaluate_swin,
    load_model=load_swin,
    force=FORCE_RETRAIN,
)
print("Accuracy en test:", round(metrics["accuracy"], 4))

## Observaciones

* Swin es una arquitectura **jerárquica** con atención local: a diferencia de
  un ViT plano, `PatchMerging` construye varias escalas (16×16 → 8×8 → 4×4 →
  2×2 tokens), algo útil cuando costa, bosque y autopista se distinguen a
  varias resoluciones (horizonte de la costa, textura del bosque, líneas de la
  calzada).
* Entrenado **desde cero** con ~612 imágenes, el Transformer no hereda los
  sesgos inductivos de las convoluciones y es el modelo con más riesgo de
  sobreajuste de los cuatro: augmentación, weight decay y early stopping no
  son opcionales aquí, son lo que evita que memorice el train.
* **Caveat honesto**: este Swin es *data-starved* frente a la DenseNet121, que
  parte de ImageNet. Si la comparación muestra un hueco entre ambos, ese hueco
  **mezcla arquitectura y preentrenamiento** (ImageNet, ~1,2 M de imágenes,
  frente a las 961 del reto) y no se puede atribuir en solitario a la atención;
  la parte pura de arquitectura se aísla mejor comparando también con la CNN
  mejorada, que también se entrena desde cero.
* El modelo se guarda como `model.keras` y también como `model.weights.h5`; la
  celda `load_swin` reconstruye la arquitectura y carga los pesos para evitar
  problemas de deserialización de las capas personalizadas. Los artefactos
  completos quedan en `resultados/swin/` y la comparación con el resto de
  modelos, en `reto-1.ipynb`.
